In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [ ]:
import random, re
from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM

from msnap.utils import common_utils, json_utils, container_utils, file_utils, model_utils, tokenizer_utils
from msnap.utils.container_utils import chunks
from msnap.core import msnap_prompts

In [ ]:
SEED = 42
common_utils.set_seed(SEED)

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/msnap'
data_dir = f'{work_dir}/data'
in_dir = f'{data_dir}/check_contexts'
out_dir = f'{data_dir}/check_targets'

# zero_shot_target = 'fact'
zero_shot_target = 'other'

out_dir = f'{out_dir}/zero_shot_{zero_shot_target}'

model_name = 'Llama-3.2-3B'
# model_name = 'Llama-3.1-8B'

model_name_or_path = f'meta-llama/{model_name}-Instruct'
dtype = 'bfloat16'
device = 'cuda:0'
max_seq_length = 4096
max_new_tokens = 64

model: AutoModelForCausalLM = model_utils.get_model(model_name_or_path, dtype, device, is_eval=True)

# 평가/추론 시에는 반드시 'left' 패딩
tokenizer: PreTrainedTokenizerFast = tokenizer_utils.load_tokenizer(model_name_or_path, 'left')

In [ ]:
in_file_path = f'{in_dir}/zero_shot_{zero_shot_target}/msnap_gpt-5.2_checked_contexts_{model_name}_{zero_shot_target}.json'
datas = json_utils.load_json(in_file_path)

print(json_utils.to_str(datas[0]))

In [ ]:
def make_prompts(datas, extract_fact_size):
    prompts_front, prompts_back, prompts_shuffle = [], [], []
    answers_fact, answers_counter = [], []
    cnt_counter = 0

    for data in datas:
        query = data['question']
        answer_fact = data['answer_fact']
        answer_counter = data['answer_counter']
        context_fact_dict: dict = data['contexts_fact']
        context_counter_dict: dict = data['contexts_counter']

        contexts_fact = list(context_fact_dict.values())
        contexts_counter = list(context_counter_dict.values())

        fact_size = len(contexts_fact)
        cnt_counter += len(contexts_counter)

        for context_counter in contexts_counter:
            if extract_fact_size <= fact_size:
                contexts_fact_extract = random.sample(contexts_fact, extract_fact_size)
            else:
                contexts_fact_extract = contexts_fact
            
            # print(f'contexts_fact_extract : {contexts_fact_extract}')
            # context_counter = 'counter'

            list_front = [context_counter] + contexts_fact_extract
            list_back = contexts_fact_extract + [context_counter]
            list_shuffle = contexts_fact_extract + [context_counter]
            random.shuffle(list_shuffle)

            # print(f'list_front : {list_front}')
            # print(f'list_back : {list_back}')
            # print(f'list_shuffle : {list_shuffle}\n\n')

            prompts_front.append(msnap_prompts.get_generate_prompt(query, list_front))
            prompts_back.append(msnap_prompts.get_generate_prompt(query, list_back))
            prompts_shuffle.append(msnap_prompts.get_generate_prompt(query, list_shuffle))

            # print(f'prompts_front : {prompts_front}')
            # print(f'prompts_back : {prompts_back}')
            # print(f'prompts_shuffle : {prompts_shuffle}')

            answers_fact.append(answer_fact)
            answers_counter.append(answer_counter)
    
    print(f'make_prompts() cnt_counter : {cnt_counter}\n')
    print(f'make_prompts() prompts_front size : {len(prompts_front)}')
    print(f'make_prompts() prompts_back size : {len(prompts_back)}')
    print(f'make_prompts() prompts_shuffle size : {len(prompts_shuffle)}')
    print(f'make_prompts() answers_fact size : {len(answers_fact)}')
    print(f'make_prompts() answers_counter size : {len(answers_counter)}\n')

    return prompts_front, prompts_back, prompts_shuffle, answers_fact, answers_counter

In [ ]:
# prompts_front, prompts_back, prompts_shuffle, answers_fact, answers_counter = make_prompts(datas, 3)

# print(f'prompts_front :\n{prompts_front[0]}\n\n')
# print(f'prompts_back :\n{prompts_back[0]}\n\n')
# print(f'prompts_shuffle :\n{prompts_shuffle[0]}\n\n')

# print(f'answers_fact : {answers_fact[0]}')
# print(f'answers_counter : {answers_counter[0]}')

In [ ]:
def check_answer(generated_text, answer_fact, answer_counter):
    generated_text = generated_text.lower().strip(' ,.!?~')
    answer_fact = answer_fact.lower().strip()
    answer_counter = answer_counter.lower().strip()

    if generated_text == answer_fact:
        return 'FACT'
    elif generated_text == answer_counter:
        return 'COUNTER'

    fact_pat = re.compile(rf'\b{re.escape(answer_fact)}\b')
    counter_pat = re.compile(rf'\b{re.escape(answer_counter)}\b')

    has_fact = bool(fact_pat.search(generated_text))
    has_counter = bool(counter_pat.search(generated_text))

    if has_fact and not has_counter:
        return 'FACT'
    elif not has_fact and has_counter:
        return 'COUNTER'
    
    return 'OTHER'

In [ ]:
def add_checked_result(results: list, prompt, generated_text, answer_fact, answer_counter):
    result = {
        'prompt': prompt,
        'generated_text': generated_text,
        'answer_fact': answer_fact,
        'answer_counter': answer_counter
    }

    results.append(result)

In [ ]:
def check_targets(prompts, answers_fact, answers_counter, batch_size, out_prefix, file_prefix):
    cnt_fact, cnt_counter, cnt_other = 0, 0, 0

    results_fact, results_counter = [], []
    other_dict = {}

    for prompts_batch, answers_fact_batch, answers_counter_batch in zip(chunks(prompts, batch_size), chunks(answers_fact, batch_size), chunks(answers_counter, batch_size)):
        # print(f'prompts_batch size : {len(prompts_batch)}')
        # print(f'answers_fact_batch size : {len(answers_fact_batch)}')
        # print(f'answers_counter_batch size : {len(answers_counter_batch)}')

        generated_texts = model_utils.get_generated_texts(
            model, tokenizer, device,
            prompts_batch, max_seq_length, max_new_tokens
        )

        # print(f'generated_texts : {len(generated_texts)}\n')
        # print(f'generated_texts : {generated_texts}')
        # print(f'answers_fact_batch : {answers_fact_batch}')
        # print(f'answers_counter_batch : {answers_counter_batch}\n')

        for i, generated_text in enumerate(generated_texts):
            prompt = prompts_batch[i]
            answer_fact = answers_fact_batch[i]
            answer_counter = answers_counter_batch[i]

            # print(f'generated_text : {generated_text}')
            # print(f'answer_fact : {answer_fact}')
            # print(f'answer_counter : {answer_counter}\n')

            checked_answer = check_answer(generated_text, answer_fact, answer_counter)

            if checked_answer == 'FACT':
                cnt_fact += 1
                add_checked_result(results_fact, prompt, generated_text, answer_fact, answer_counter)
            elif checked_answer == 'COUNTER':
                cnt_counter += 1
                add_checked_result(results_counter, prompt, generated_text, answer_fact, answer_counter)
            else:
                cnt_other += 1
                
                other_key = f'{answer_fact}\t{answer_counter}\t{generated_text}'
                container_utils.add_str_int(other_dict, other_key, 1)

        # break

    cnt_all = cnt_fact + cnt_counter + cnt_other
    print(f'check_targets() cnt_all : {cnt_all}')
    print(f'check_targets() cnt_fact : {cnt_fact} ({(cnt_fact/cnt_all):.4f})')
    print(f'check_targets() cnt_counter : {cnt_counter} ({(cnt_counter/cnt_all):.4f})')
    print(f'check_targets() cnt_other : {cnt_other} ({(cnt_other/cnt_all):.4f})\n')

    json_utils.write_json(results_fact, f'{out_prefix}/fact/{file_prefix}_answer-fact.json')
    json_utils.write_jsonl(results_fact, f'{out_prefix}/fact/{file_prefix}_answer-fact.jsonl')

    json_utils.write_json(results_counter, f'{out_prefix}/counter/{file_prefix}_answer-counter.json')
    json_utils.write_jsonl(results_counter, f'{out_prefix}/counter/{file_prefix}_answer-counter.jsonl')

    file_utils.write_dict(container_utils.sorted_dict(other_dict), f'{out_prefix}/other/{file_prefix}_answer-other.dict')
    print()

In [ ]:
for extract_fact_size in [0, 1, 2, 3, 4, 5]:
    prompts_front, prompts_back, prompts_shuffle, answers_fact, answers_counter = make_prompts(datas, extract_fact_size)

    out_prefix = f'{out_dir}/{model_name}'
    file_prefix = f'msnap_gpt-5.2_checked_targets_{model_name}_fact-{extract_fact_size}_counter'
    print(f'# out_prefix : {out_prefix}')
    print(f'# file_prefix : {file_prefix}\n')

    check_targets(prompts_front, answers_fact, answers_counter, 20, out_prefix, f'{file_prefix}-front')
    check_targets(prompts_back, answers_fact, answers_counter, 20, out_prefix, f'{file_prefix}-back')
    check_targets(prompts_shuffle, answers_fact, answers_counter, 20, out_prefix, f'{file_prefix}-shuffle')